# 03 — Data Enrichment: Market Data, Technicals & News

This notebook enriches your portfolio holdings with:
- **Current & historical prices** (yfinance)
- **Technical indicators** — RSI, SMA 50/200, EMA 12/26, MACD, Bollinger Bands (pandas-ta)
- **Fundamentals** — P/E, market cap, dividend yield, 52-week range, sector (yfinance)
- **Recent news headlines** (yfinance)

The output is an enriched JSON file ready to feed into Claude for analysis in Phase 3.

**Prerequisites:**
- Run notebooks 01 and 02 first
- `portfolio_snapshot.json` must exist
- Run `uv sync` after pulling this update (new dependencies: yfinance, pandas-ta)

## Setup

In [1]:
import json
import os
import warnings
from datetime import datetime, timedelta, timezone

import pandas as pd
import pandas_ta as ta
import yfinance as yf

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

# Load portfolio snapshot from notebook 02
SNAPSHOT_FILE = os.path.join("..", "portfolio_snapshot.json")

if not os.path.exists(SNAPSHOT_FILE):
    raise FileNotFoundError(
        "portfolio_snapshot.json not found. Run notebook 02 first."
    )

with open(SNAPSHOT_FILE, "r") as f:
    portfolio = json.load(f)

holdings_df = pd.DataFrame(portfolio["holdings"])

# Get unique tickers (exclude cash and None)
tickers = (
    holdings_df[holdings_df["type"] != "cash"]["ticker"]
    .dropna()
    .unique()
    .tolist()
)

# Add RSU tickers for price tracking even if not currently held
RSU_TICKERS = ["SNOW"]
for t in RSU_TICKERS:
    if t not in tickers:
        tickers.append(t)
        print(f"📌 Added RSU ticker {t} for price tracking")

print(f"✅ Loaded portfolio: {len(holdings_df)} positions")
print(f"📊 Unique tickers to enrich: {len(tickers)}")
print(f"   {tickers}")
def format_yf_option(ticker: str) -> str:
    """Convert Fidelity-style tickers like -JD270115C25 to yfinance-style JD270115C00025000."""
    import re
    # Expected format: -[TICKER][YYMMDD][C|P][STRIKE]
    match = re.match(r"-?([A-Z]+)(\d{6})([CP])(\d+(\.\d+)?)", ticker)
    if not match: return ticker
    symbol, date, cp, strike, _ = match.groups()
    # Strike must be 8 digits (xxxxx.yyy -> xxxxxxxx)
    strike_val = float(strike)
    strike_str = f"{int(strike_val * 1000):08d}"
    return f"{symbol}{date}{cp}{strike_str}"


📌 Added RSU ticker SNOW for price tracking
✅ Loaded portfolio: 15 positions
📊 Unique tickers to enrich: 15
   ['UPS', 'META', 'NVO', 'LULU', 'BABA', 'TSLA', 'ZETA', 'GRAB', 'O', 'AHRT', 'VICI', '-JD270115C25', '-JD270617C25', '-NVO270115C30', 'SNOW']


## 1. Fetch Historical Prices (6 months daily)

In [2]:
LOOKBACK_DAYS = 200  # enough for SMA-200
start_date = (datetime.now() - timedelta(days=LOOKBACK_DAYS)).strftime("%Y-%m-%d")

print(f"Downloading daily prices since {start_date}...\n")

price_data = {}  # ticker -> DataFrame
failed_tickers = []

for ticker in tickers:
    try:
        df = yf.download(format_yf_option(ticker), start=start_date, progress=False, auto_adjust=True)
        if df.empty:
            print(f"  ⚠️  {ticker}: No data returned (may be delisted or invalid)")
            failed_tickers.append(ticker)
        else:
            # Flatten multi-level columns if present
            if isinstance(df.columns, pd.MultiIndex):
                df.columns = df.columns.get_level_values(0)
            # Drop rows with NaN close prices (partial/settling day data)
            df = df[df["Close"].notna()]
            if df.empty:
                print(f"  ⚠️  {ticker}: All close prices are NaN")
                failed_tickers.append(ticker)
                continue
            price_data[ticker] = df
            print(f"  ✅ {ticker}: {len(df)} days, latest close ${df['Close'].iloc[-1]:.2f}")
    except Exception as e:
        print(f"  ❌ {ticker}: {e}")
        failed_tickers.append(ticker)

print(f"\n📈 Price data fetched for {len(price_data)}/{len(tickers)} tickers")
if failed_tickers:
    print(f"⚠️  Failed/skipped: {failed_tickers}")

  ✅ UPS: 138 days, latest close $96.56
  ✅ META: 138 days, latest close $606.70


  ✅ NVO: 138 days, latest close $37.08


  ✅ LULU: 138 days, latest close $165.57


  ✅ BABA: 138 days, latest close $124.90


  ✅ TSLA: 138 days, latest close $380.30


...

  ✅ GRAB: 138 days, latest close $3.68


  ✅ O: 138 days, latest close $62.64


  ✅ AHRT: 138 days, latest close $6.02
  ✅ VICI: 138 days, latest close $27.98


  ✅ -JD270115C25: 130 days, latest close $5.35
  ✅ -JD270617C25: 25 days, latest close $6.65


  ✅ -NVO270115C30: 129 days, latest close $9.88
  ✅ SNOW: 138 days, latest close $175.40

📈 Price data fetched for 15/15 tickers


## 2. Calculate Technical Indicators

For each ticker, we compute:
- **SMA 50 / SMA 200** — trend (Golden Cross / Death Cross)
- **EMA 12 / EMA 26** — momentum
- **RSI (14)** — overbought (>70) / oversold (<30)
- **MACD** — trend direction and momentum
- **Bollinger Bands (20, 2)** — volatility and mean reversion

In [3]:
technicals = {}  # ticker -> dict of latest indicator values

for ticker, df in price_data.items():
    close = df["Close"]
    
    # Moving averages
    sma_50 = ta.sma(close, length=50)
    sma_200 = ta.sma(close, length=200)
    ema_12 = ta.ema(close, length=12)
    ema_26 = ta.ema(close, length=26)
    
    # RSI
    rsi = ta.rsi(close, length=14)
    
    # MACD
    macd_df = ta.macd(close, fast=12, slow=26, signal=9)
    
    # Bollinger Bands
    bbands = ta.bbands(close, length=20, std=2)
    
    latest_price = close.iloc[-1]
    
    tech = {
        "price": round(float(latest_price), 2),
        "sma_50": round(float(sma_50.iloc[-1]), 2) if sma_50 is not None and not sma_50.empty and pd.notna(sma_50.iloc[-1]) else None,
        "sma_200": round(float(sma_200.iloc[-1]), 2) if sma_200 is not None and not sma_200.empty and pd.notna(sma_200.iloc[-1]) else None,
        "ema_12": round(float(ema_12.iloc[-1]), 2) if ema_12 is not None and not ema_12.empty else None,
        "ema_26": round(float(ema_26.iloc[-1]), 2) if ema_26 is not None and not ema_26.empty else None,
        "rsi_14": round(float(rsi.iloc[-1]), 2) if rsi is not None and not rsi.empty and pd.notna(rsi.iloc[-1]) else None,
    }
    
    # MACD values
    if macd_df is not None and not macd_df.empty:
        cols = macd_df.columns
        tech["macd"] = round(float(macd_df[cols[0]].iloc[-1]), 4) if pd.notna(macd_df[cols[0]].iloc[-1]) else None
        tech["macd_signal"] = round(float(macd_df[cols[2]].iloc[-1]), 4) if pd.notna(macd_df[cols[2]].iloc[-1]) else None
        tech["macd_hist"] = round(float(macd_df[cols[1]].iloc[-1]), 4) if pd.notna(macd_df[cols[1]].iloc[-1]) else None
    
    # Bollinger Bands
    if bbands is not None and not bbands.empty:
        bb_cols = bbands.columns
        tech["bb_lower"] = round(float(bbands[bb_cols[0]].iloc[-1]), 2) if pd.notna(bbands[bb_cols[0]].iloc[-1]) else None
        tech["bb_mid"] = round(float(bbands[bb_cols[1]].iloc[-1]), 2) if pd.notna(bbands[bb_cols[1]].iloc[-1]) else None
        tech["bb_upper"] = round(float(bbands[bb_cols[2]].iloc[-1]), 2) if pd.notna(bbands[bb_cols[2]].iloc[-1]) else None
    
    # Derived signals
    if tech["sma_50"] and tech["sma_200"]:
        tech["golden_cross"] = tech["sma_50"] > tech["sma_200"]
    if tech["rsi_14"]:
        tech["rsi_signal"] = (
            "overbought" if tech["rsi_14"] > 70
            else "oversold" if tech["rsi_14"] < 30
            else "neutral"
        )
    if tech.get("bb_lower") and tech.get("bb_upper"):
        tech["bb_position"] = (
            "below_lower" if latest_price < tech["bb_lower"]
            else "above_upper" if latest_price > tech["bb_upper"]
            else "within_bands"
        )
    
    # Price vs moving averages
    if tech["sma_50"]:
        tech["pct_above_sma50"] = round((latest_price / tech["sma_50"] - 1) * 100, 2)
    if tech["sma_200"]:
        tech["pct_above_sma200"] = round((latest_price / tech["sma_200"] - 1) * 100, 2)
    
    technicals[ticker] = tech

print(f"✅ Technicals computed for {len(technicals)} tickers")

✅ Technicals computed for 15 tickers


### Technicals Overview Table

In [4]:
tech_df = pd.DataFrame.from_dict(technicals, orient="index")
tech_df.index.name = "ticker"

# Show key columns
display_cols = ["price", "sma_50", "sma_200", "rsi_14", "rsi_signal", "macd_hist", "bb_position"]
available_cols = [c for c in display_cols if c in tech_df.columns]

tech_df[available_cols]

,price,sma_50,sma_200,rsi_14,rsi_signal,macd_hist,bb_position
ticker,,,,,,,
UPS,96.56,108.33,None,26.43,oversold,-1.2608,within_bands
META,606.70,650.60,None,36.05,neutral,-4.2436,below_lower
NVO,37.08,48.71,None,31.82,neutral,0.4208,within_bands
LULU,165.57,181.06,None,42.28,neutral,-0.2567,within_bands
BABA,124.90,154.14,None,25.22,oversold,-0.1021,within_bands
TSLA,380.30,416.55,None,36.43,neutral,-0.6039,below_lower
ZETA,17.23,18.71,None,44.69,neutral,-0.0238,within_bands
GRAB,3.68,4.28,None,31.41,neutral,-0.0147,within_bands
O,62.64,62.94,None,39.00,neutral,-0.5243,below_lower


## 3. Fetch Fundamentals

In [5]:
fundamentals = {}  # ticker -> dict

for ticker in price_data.keys():
    try:
        info = yf.Ticker(format_yf_option(ticker)).info
        
        fundamentals[ticker] = {
            "sector": info.get("sector"),
            "industry": info.get("industry"),
            "market_cap": info.get("marketCap"),
            "pe_trailing": info.get("trailingPE"),
            "pe_forward": info.get("forwardPE"),
            "peg_ratio": info.get("pegRatio"),
            "price_to_book": info.get("priceToBook"),
            "dividend_yield": info.get("dividendYield"),
            "beta": info.get("beta"),
            "fifty_two_week_high": info.get("fiftyTwoWeekHigh"),
            "fifty_two_week_low": info.get("fiftyTwoWeekLow"),
            "avg_volume": info.get("averageVolume"),
            "revenue_growth": info.get("revenueGrowth"),
            "earnings_growth": info.get("earningsGrowth"),
            "profit_margin": info.get("profitMargins"),
            "return_on_equity": info.get("returnOnEquity"),
            "debt_to_equity": info.get("debtToEquity"),
            "free_cash_flow": info.get("freeCashflow"),
            "analyst_target": info.get("targetMeanPrice"),
            "analyst_rec": info.get("recommendationKey"),
            "short_description": info.get("longBusinessSummary", "")[:200],
        }
        print(f"  ✅ {ticker}: {fundamentals[ticker].get('sector', 'N/A')} | P/E {fundamentals[ticker].get('pe_trailing', 'N/A')}")
    except Exception as e:
        print(f"  ⚠️  {ticker}: {e}")

print(f"\n✅ Fundamentals fetched for {len(fundamentals)}/{len(price_data)} tickers")

  ✅ UPS: Industrials | P/E 14.719512


  ✅ META: Communication Services | P/E 25.839012


  ✅ NVO: Healthcare | P/E 10.415731


  ✅ LULU: Consumer Cyclical | P/E 11.505907


  ✅ BABA: Consumer Cyclical | P/E 16.455862


...

  ✅ VICI: Real Estate | P/E 10.720306


  ✅ -JD270115C25: None | P/E None


  ✅ -JD270617C25: None | P/E None


  ✅ -NVO270115C30: None | P/E None


  ✅ SNOW: Technology | P/E None

✅ Fundamentals fetched for 15/15 tickers


In [6]:
fund_df = pd.DataFrame.from_dict(fundamentals, orient="index")
fund_df.index.name = "ticker"

display_cols = ["sector", "market_cap", "pe_trailing", "pe_forward", "dividend_yield", "beta", "analyst_rec", "analyst_target"]
available_cols = [c for c in display_cols if c in fund_df.columns]

fund_df[available_cols]

,sector,market_cap,pe_trailing,pe_forward,dividend_yield,beta,analyst_rec,analyst_target
ticker,,,,,,,,
UPS,Industrials,8.198926e+10,14.719512,12.130135,6.79,1.054,buy,113.071430
META,Communication Services,1.534681e+12,25.839012,16.909142,0.35,1.279,strong_buy,863.629330
NVO,Healthcare,1.652909e+11,10.415731,11.181086,5.01,0.265,buy,47.328884
LULU,Consumer Cyclical,1.963481e+10,11.505907,12.427773,NaN,1.013,hold,191.924580
BABA,Consumer Cyclical,2.981878e+11,16.455862,15.065073,0.84,NaN,strong_buy,198.705400
TSLA,Consumer Cyclical,1.427050e+12,358.773600,135.317370,NaN,1.926,buy,421.613650
ZETA,Technology,4.237139e+09,NaN,14.504466,NaN,1.280,buy,29.083330
GRAB,Technology,1.508977e+10,61.333336,25.136612,NaN,0.962,strong_buy,6.495380
O,Real Estate,5.861694e+10,53.538464,35.678890,5.16,0.774,hold,67.850000


## 4. Fetch Recent News

In [7]:
MAX_NEWS_PER_TICKER = 5

news_data = {}  # ticker -> list of headlines

for ticker in price_data.keys():
    try:
        yf_ticker = yf.Ticker(format_yf_option(ticker))
        raw_news = yf_ticker.news or []
        
        headlines = []
        for item in raw_news[:MAX_NEWS_PER_TICKER]:
            headline = {
                "title": item.get("title", ""),
                "publisher": item.get("publisher", ""),
                "link": item.get("link", ""),
            }
            # Extract publish time if available
            pub_time = item.get("providerPublishTime")
            if pub_time:
                headline["published"] = datetime.fromtimestamp(pub_time).strftime("%Y-%m-%d %H:%M")
            headlines.append(headline)
        
        news_data[ticker] = headlines
        print(f"  ✅ {ticker}: {len(headlines)} articles")
    except Exception as e:
        print(f"  ⚠️  {ticker}: {e}")
        news_data[ticker] = []

print(f"\n📰 News fetched for {len(news_data)} tickers")


  ✅ UPS: 5 articles


  ✅ META: 5 articles
  ✅ NVO: 5 articles


  ✅ LULU: 5 articles


  ✅ BABA: 5 articles
  ✅ TSLA: 5 articles


  ✅ ZETA: 5 articles


...

  ✅ O: 5 articles


  ✅ AHRT: 0 articles


  ✅ VICI: 5 articles
  ✅ -JD270115C25: 0 articles


  ✅ -JD270617C25: 0 articles
  ✅ -NVO270115C30: 0 articles


  ✅ SNOW: 5 articles

📰 News fetched for 15 tickers


In [8]:
# Preview news for first ticker
preview_ticker = list(news_data.keys())[0] if news_data else None

if preview_ticker and news_data[preview_ticker]:
    print(f"\n📰 Recent headlines for {preview_ticker}:")
    for article in news_data[preview_ticker]:
        date = article.get('published', 'N/A')
        print(f"  [{date}] {article['title']}")
        print(f"           — {article['publisher']}")
else:
    print("No news available to preview.")


📰 Recent headlines for UPS:
  [N/A] 
           — 
  [N/A] 
           — 
  [N/A] 
           — 
  [N/A] 
           — 
  [N/A] 
           — 


## 5. Performance Metrics

In [9]:
performance = {}  # ticker -> return periods

for ticker, df in price_data.items():
    close = df["Close"]
    latest = close.iloc[-1]
    
    perf = {}
    
    # 1-day return
    if len(close) >= 2:
        perf["return_1d"] = round(float((latest / close.iloc[-2] - 1) * 100), 2)
    
    # 1-week return
    if len(close) >= 5:
        perf["return_1w"] = round(float((latest / close.iloc[-5] - 1) * 100), 2)
    
    # 1-month return
    if len(close) >= 21:
        perf["return_1m"] = round(float((latest / close.iloc[-21] - 1) * 100), 2)
    
    # 3-month return
    if len(close) >= 63:
        perf["return_3m"] = round(float((latest / close.iloc[-63] - 1) * 100), 2)
    
    # 6-month return
    if len(close) >= 126:
        perf["return_6m"] = round(float((latest / close.iloc[-126] - 1) * 100), 2)
    
    # 52-week high/low proximity
    high = fundamentals.get(ticker, {}).get("fifty_two_week_high")
    low = fundamentals.get(ticker, {}).get("fifty_two_week_low")
    if high:
        perf["pct_from_52w_high"] = round(float((latest / high - 1) * 100), 2)
    if low:
        perf["pct_from_52w_low"] = round(float((latest / low - 1) * 100), 2)
    
    # Analyst target upside/downside
    target = fundamentals.get(ticker, {}).get("analyst_target")
    if target:
        perf["analyst_upside_pct"] = round(float((target / latest - 1) * 100), 2)
    
    performance[ticker] = perf

perf_df = pd.DataFrame.from_dict(performance, orient="index")
perf_df.index.name = "ticker"

print("📊 Performance Summary:")
perf_df

📊 Performance Summary:


,return_1d,return_1w,return_1m,return_3m,return_6m,pct_from_52w_high,pct_from_52w_low,analyst_upside_pct
ticker,,,,,,,,
UPS,-0.29,-0.67,-16.43,-3.02,17.12,-21.12,17.76,17.10
META,-1.46,-1.06,-5.83,-6.51,-22.06,-23.81,26.45,42.35
NVO,-0.99,-2.32,-23.47,-22.38,-40.05,-54.47,3.43,27.64
LULU,0.11,4.94,-9.47,-20.35,-2.47,-52.49,5.70,15.92
BABA,-7.09,-7.63,-19.04,-15.09,-23.13,-35.17,30.47,59.09
TSLA,-3.18,-2.79,-7.63,-18.61,-8.77,-23.76,77.50,10.86
ZETA,0.06,-3.42,9.89,-0.92,-18.34,-30.80,61.18,68.79
GRAB,-1.87,-0.81,-16.36,-24.44,-42.05,-44.41,9.52,76.50
O,-0.63,-2.79,-3.98,9.90,8.89,-7.80,23.53,8.32


## 6. Export Enriched Portfolio for Claude (Phase 3)

We combine everything into a single JSON file:
- Portfolio holdings (from Plaid)
- Technicals per ticker
- Fundamentals per ticker
- Performance metrics per ticker
- Recent news per ticker

In [10]:
enriched_portfolio = {
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "summary": portfolio["summary"],
    "holdings": portfolio["holdings"],
    "enrichment": {},
    "failed_tickers": failed_tickers,
}

# Combine all enrichment data per ticker
for ticker in price_data.keys():
    enriched_portfolio["enrichment"][ticker] = {
        "technicals": technicals.get(ticker, {}),
        "fundamentals": fundamentals.get(ticker, {}),
        "performance": performance.get(ticker, {}),
        "news": news_data.get(ticker, []),
    }

# Save
export_path = os.path.join("..", "enriched_portfolio.json")
with open(export_path, "w") as f:
    json.dump(enriched_portfolio, f, indent=2, default=str)

# File size
size_kb = os.path.getsize(export_path) / 1024

print(f"💾 Enriched portfolio saved to {export_path} ({size_kb:.1f} KB)")
print(f"")
print(f"   📋 Holdings:     {len(enriched_portfolio['holdings'])} positions")
print(f"   📈 Enriched:     {len(enriched_portfolio['enrichment'])} tickers")
print(f"   🔧 Technicals:   RSI, SMA 50/200, EMA, MACD, Bollinger Bands")
print(f"   📊 Fundamentals: P/E, market cap, beta, analyst targets, margins")
print(f"   📰 News:         {sum(len(v) for v in news_data.values())} total articles")
print(f"")
print(f"👉 This file will be fed to Claude for analysis in notebook 04 (Phase 3).")


💾 Enriched portfolio saved to ../enriched_portfolio.json (37.0 KB)

   📋 Holdings:     15 positions
   📈 Enriched:     15 tickers
   🔧 Technicals:   RSI, SMA 50/200, EMA, MACD, Bollinger Bands
   📊 Fundamentals: P/E, market cap, beta, analyst targets, margins
   📰 News:         55 total articles

👉 This file will be fed to Claude for analysis in notebook 04 (Phase 3).


## Quick Flags (Preview of What Claude Will Analyze)

A quick scan for positions that might warrant attention:

In [11]:
print("🚩 POSITIONS WORTH WATCHING:\n")

for ticker, tech in technicals.items():
    flags = []
    
    # RSI extremes
    if tech.get("rsi_signal") == "overbought":
        flags.append(f"⚠️  RSI {tech['rsi_14']} — overbought")
    elif tech.get("rsi_signal") == "oversold":
        flags.append(f"🟢 RSI {tech['rsi_14']} — oversold (potential buy)")
    
    # Death cross
    if tech.get("golden_cross") is False:
        flags.append("☠️  Death Cross (SMA 50 < SMA 200)")
    
    # Bollinger Band breakout
    if tech.get("bb_position") == "below_lower":
        flags.append("📉 Below lower Bollinger Band")
    elif tech.get("bb_position") == "above_upper":
        flags.append("📈 Above upper Bollinger Band")
    
    # Near 52-week low
    perf = performance.get(ticker, {})
    if perf.get("pct_from_52w_low") is not None and perf["pct_from_52w_low"] < 5:
        flags.append(f"🔻 Near 52-week low ({perf['pct_from_52w_low']:+.1f}%)")
    
    # Near 52-week high
    if perf.get("pct_from_52w_high") is not None and perf["pct_from_52w_high"] > -5:
        flags.append(f"🔺 Near 52-week high ({perf['pct_from_52w_high']:+.1f}%)")
    
    # Analyst upside/downside
    if perf.get("analyst_upside_pct") is not None:
        if perf["analyst_upside_pct"] > 20:
            flags.append(f"📊 Analyst target: {perf['analyst_upside_pct']:+.1f}% upside")
        elif perf["analyst_upside_pct"] < -10:
            flags.append(f"📊 Analyst target: {perf['analyst_upside_pct']:+.1f}% downside")
    
    if flags:
        print(f"  {ticker}:")
        for flag in flags:
            print(f"    {flag}")
        print()

print("\n💡 Claude will analyze these signals in context with fundamentals and news in Phase 3.")

🚩 POSITIONS WORTH WATCHING:

  UPS:
    🟢 RSI 26.43 — oversold (potential buy)

  META:
    📉 Below lower Bollinger Band
    📊 Analyst target: +42.4% upside

  NVO:
    🔻 Near 52-week low (+3.4%)
    📊 Analyst target: +27.6% upside

  BABA:
    🟢 RSI 25.22 — oversold (potential buy)
    📊 Analyst target: +59.1% upside

  TSLA:
    📉 Below lower Bollinger Band

  ZETA:
    📊 Analyst target: +68.8% upside

  GRAB:
    📊 Analyst target: +76.5% upside

  O:
    📉 Below lower Bollinger Band

  AHRT:
    📊 Analyst target: +20.4% upside

  VICI:
    🔻 Near 52-week low (+1.8%)
    📊 Analyst target: +24.3% upside

  -JD270115C25:
    🔻 Near 52-week low (+4.9%)
    🔺 Near 52-week high (-0.0%)

  -JD270617C25:
    🔻 Near 52-week low (+0.0%)
    🔺 Near 52-week high (+0.0%)

  -NVO270115C30:
    🔻 Near 52-week low (+4.5%)
    🔺 Near 52-week high (+0.0%)

  SNOW:
    📊 Analyst target: +36.7% upside


💡 Claude will analyze these signals in context with fundamentals and news in Phase 3.
